# 02 — 策略对比：各合约费率特征 & 最优组合

本 notebook 回答：
1. **每个合约单独跑有多好？** — 从中挑出表现最强的
2. **多合约组合 vs 单合约** — 多样化是否真的有效？
3. **冷却期对各合约的影响** — 高频交易合约（MSTR/COIN）受益最大
4. **滚动均值信号 vs 实时信号** — 稳定性权衡

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data.binance import load_all, EQUITY_SYMBOLS
from src.backtest.engine import run
from src.backtest.costs import DEFAULT
from src.backtest.metrics import summary, print_summary

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

wide = load_all()
print(f'数据加载完成: {wide.shape}  ({len(wide)/3:.0f}天)')

## 1. 各合约单独回测

In [ ]:
single_results = {}
for sym in wide.columns:
    col = wide[[sym]].copy()
    if (col != 0).sum().sum() < 10:
        continue
    eq, trades = run(col, cost=DEFAULT, threshold=0.0001, max_leverage=5.0)
    s = summary(eq, label=sym.replace('USDT', ''))
    s['n_trades'] = len(trades)
    single_results[sym] = (eq, s)

# 汇总表
rows = [s for _, (_, s) in single_results.items()]
df_single = pd.DataFrame(rows).set_index('label')
df_single.sort_values('ann_ret_%', ascending=False)

In [ ]:
# 各合约净值曲线
fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharex=True)
axes_flat = axes.flatten()

for idx, (sym, (eq, s)) in enumerate(single_results.items()):
    if idx >= len(axes_flat):
        break
    ax = axes_flat[idx]
    ax.plot(eq.index, eq.values, lw=1.2, color='steelblue')
    ax.axhline(1, color='gray', ls='--', lw=0.6)
    ax.set_title(f"{s['label']}\n年化={s['ann_ret_%']:+.0f}% | 夏普={s['sharpe']:.2f}")
    ax.tick_params(axis='x', rotation=30)

# 隐藏多余子图
for ax in axes_flat[len(single_results):]:
    ax.set_visible(False)

plt.suptitle('各合约单独回测净值曲线', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 2. 组合对比（全部 vs 仅稳定合约）

In [ ]:
# 「稳定合约」= 年化费率正/负占比 > 80%（方向一致性高）
ann = wide * 3 * 365 * 100
pos_ratio = (wide > 0).mean()
neg_ratio = (wide < 0).mean()
stable = pos_ratio[(pos_ratio > 0.8) | (neg_ratio > 0.8)].index.tolist()
volatile = [s for s in wide.columns if s not in stable]

print('稳定合约（方向一致性 > 80%）:', [s.replace('USDT','') for s in stable])
print('波动合约（方向多变）        :', [s.replace('USDT','') for s in volatile])

portfolios = {
    '全部合约': wide,
    '稳定合约': wide[stable] if stable else None,
    '波动合约': wide[volatile] if volatile else None,
}

portfolio_eq = {}
for name, subset in portfolios.items():
    if subset is None or len(subset.columns) == 0:
        continue
    eq, _ = run(subset, cost=DEFAULT, threshold=0.0001, max_leverage=5.0)
    portfolio_eq[name] = eq
    print_summary(eq, label=name)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['steelblue', 'green', 'red']
for (name, eq), color in zip(portfolio_eq.items(), colors):
    ax.plot(eq.index, eq.values, label=name, color=color, lw=1.5)
ax.axhline(1, color='gray', ls='--', lw=0.6)
ax.set_title('全部合约 vs 稳定合约 vs 波动合约 组合净值对比')
ax.set_ylabel('净值')
ax.legend()
plt.tight_layout()
plt.show()

## 3. 冷却期效果分析

In [ ]:
from src.backtest.costs import CostModel

cooldown_results = {}
for cd in [0, 1, 2, 3, 5]:
    cost_cd = CostModel(cooldown_periods=cd)
    eq, trades = run(wide, cost=cost_cd, threshold=0.0001, max_leverage=5.0)
    s = summary(eq, label=f'冷却={cd}期')
    s['n_trades'] = len(trades)
    cooldown_results[cd] = (eq, s)
    print_summary(eq, label=f'冷却期={cd}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cd_vals = list(cooldown_results.keys())
ann_rets = [s['ann_ret_%'] for _, (_, s) in cooldown_results.items()]
sharpes  = [s['sharpe']   for _, (_, s) in cooldown_results.items()]
n_trades = [s['n_trades'] for _, (_, s) in cooldown_results.items()]

axes[0].plot(cd_vals, ann_rets, 'bo-'); axes[0].set_title('年化收益率 %'); axes[0].set_xlabel('冷却期数')
axes[1].plot(cd_vals, sharpes,  'ro-'); axes[1].set_title('夏普比率');    axes[1].set_xlabel('冷却期数')
axes[2].plot(cd_vals, n_trades, 'go-'); axes[2].set_title('开仓次数');    axes[2].set_xlabel('冷却期数')

plt.suptitle('冷却期 vs 策略表现', fontsize=12)
plt.tight_layout()
plt.show()

## 4. 滚动均值信号 vs 实时信号

In [ ]:
from src.signal.generator import generate

# 对比实时vs滚动均值的信号稳定性
results_compare = {}
for label, use_rolling in [('实时信号', False), ('7日滚动均值', True)]:
    # 构建信号（在引擎外单独生成，仅做对比）
    sig_count = 0
    dir_flips = {sym: 0 for sym in wide.columns}
    prev_dir  = {sym: 0 for sym in wide.columns}
    
    if use_rolling:
        rate_src = wide.rolling(21).mean()
    else:
        rate_src = wide
    
    for i in range(len(wide)):
        row = rate_src.iloc[i:i+1]
        sigs = generate(row, threshold=0.0001, use_rolling=False)
        for sym, (d, w, r) in sigs.items():
            if prev_dir[sym] != 0 and prev_dir[sym] != d:
                dir_flips[sym] += 1
            prev_dir[sym] = d
            sig_count += 1
    
    results_compare[label] = dir_flips

# 对比方向翻转次数
df_flip = pd.DataFrame(results_compare).rename(index=lambda x: x.replace('USDT',''))
df_flip.sort_values('实时信号', ascending=False)

In [ ]:
x = range(len(df_flip))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar([i - width/2 for i in x], df_flip['实时信号'],   width=width, label='实时信号',   color='steelblue', alpha=0.85)
ax.bar([i + width/2 for i in x], df_flip['7日滚动均值'], width=width, label='7日滚动均值', color='orange',    alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(df_flip.index)
ax.set_title('各合约信号方向翻转次数对比（实时 vs 滚动均值）')
ax.set_ylabel('翻转次数')
ax.legend()
plt.tight_layout()
plt.show()

## 5. 总结

In [ ]:
print('='*60)
print('各合约回测汇总')
print('='*60)
df_final = df_single.sort_values('ann_ret_%', ascending=False)
for idx, row in df_final.iterrows():
    stars = '★' * min(5, max(0, int(row['sharpe'])))
    print(f"  {idx:8s}  年化={row['ann_ret_%']:+6.1f}%  "
          f"夏普={row['sharpe']:5.2f}  回撤={row['max_dd_%']:5.1f}%  "
          f"开仓={row['n_trades']:3.0f}次  {stars}")